# 중앙대 반경 지도 시각화 셀 모음

이 노트북은 중앙대학교 주변을 포함해 임의의 기준점과 반경으로 재사용할 수 있는 지도 시각화 관련 셀과 결과만 모은 것입니다. 특정 장소명에 종속되지 않도록 지도 라벨과 설명을 일반화했습니다.

현재 파라미터 값은 원본 분석 설정을 따르며, 첫 번째 코드 셀의 `CENTER_LAT`, `CENTER_LON`, `RADIUS_M`를 수정해 다른 영역을 시각화할 수 있습니다.

## 1. 지도 시각화 환경과 기준 파라미터

지도에 사용할 라이브러리와 원천 데이터 경로를 준비합니다. `CENTER_LAT`, `CENTER_LON`, `RADIUS_M` 값을 바꾸면 다른 기준점과 반경으로 재사용할 수 있습니다.

In [1]:
from pathlib import Path
import json

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import pandas as pd
from IPython.display import display
from shapely.geometry import Point, shape

# Windows 한글 그래프 폰트 설정 (맑은 고딕)
rcParams['font.family'] = 'Malgun Gothic'
rcParams['axes.unicode_minus'] = False

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data').exists():
    # Jupyter를 상위 폴더에서 실행한 경우를 대비한 보정
    PROJECT_DIR = PROJECT_DIR / 'DartB_TOYPROJECT'

DATASETS = {
    'audible_signal': PROJECT_DIR / '음향신호기_현황.csv',
    'facility': PROJECT_DIR / '서울시_장애인편의시설_목록정보(한국사회보장정보원)_.csv',
    'subway_voice': PROJECT_DIR / '서울교통공사_지하철_시각장애인_음성유도기_설치_위치_정보_20260213.csv',
    'accident': PROJECT_DIR / '보행자_사고_다발지역.csv',
}

# 기존 수집/스냅 단계와 동일한 중심점 및 분석 반경
CENTER_LAT, CENTER_LON = 37.50457, 126.95696
RADIUS_M = 800

for name, path in DATASETS.items():
    print(f'{name}: {path.name} | exists={path.exists()}')

audible_signal: 음향신호기_현황.csv | exists=True
facility: 서울시_장애인편의시설_목록정보(한국사회보장정보원)_.csv | exists=True
subway_voice: 서울교통공사_지하철_시각장애인_음성유도기_설치_위치_정보_20260213.csv | exists=True
accident: 보행자_사고_다발지역.csv | exists=True


## 2. 원천 데이터 읽기 함수

공공데이터의 인코딩 차이를 처리하면서 CSV를 읽고 기본 요약을 출력하는 공통 함수를 정의합니다.

In [2]:
def read_csv_flexible(path, **kwargs):
    """UTF-8/CP949 계열 CSV를 순서대로 시도해 읽는다."""
    errors = []
    for encoding in ('utf-8-sig', 'utf-8', 'cp949', 'euc-kr'):
        try:
            frame = pd.read_csv(path, encoding=encoding, **kwargs)
            print(f'{path.name}: encoding={encoding}, rows={len(frame):,}, cols={len(frame.columns)}')
            return frame
        except UnicodeDecodeError as exc:
            errors.append(f'{encoding}: {exc}')
    raise UnicodeDecodeError('unknown', b'', 0, 1, '\n'.join(errors))

def profile(frame, name, preview_rows=3):
    """행·열 수, 자료형, 결측치, 중복 행과 표본을 빠르게 확인한다."""
    print(f'\n[{name}] shape={frame.shape}, duplicate rows={frame.duplicated().sum():,}')
    summary = pd.DataFrame({
        'dtype': frame.dtypes.astype(str),
        'missing_n': frame.isna().sum(),
        'missing_pct': (frame.isna().mean() * 100).round(2),
        'nunique': frame.nunique(dropna=True),
    }).sort_values(['missing_pct', 'nunique'], ascending=[False, False])
    display(summary)
    display(frame.head(preview_rows))
    return summary

## 3. 음향 신호기 데이터 준비

지도에 표시할 음향 신호기 원천 데이터를 불러옵니다.

In [3]:
# 1) 음향신호기 현황
audible = read_csv_flexible(DATASETS['audible_signal'])
audible_summary = profile(audible, '음향신호기')

음향신호기_현황.csv: encoding=utf-8-sig, rows=21,451, cols=22

[음향신호기] shape=(21451, 22), duplicate rows=0


,dtype,missing_n,missing_pct,nunique
MNG_AGEN,float64,21243,99.03,2
SL_NUM,str,18485,86.17,68
DRN_CDE,float64,9314,43.42,8
POS,float64,9314,43.42,1
geometry,str,4604,21.46,16847
YCE,float64,4604,21.46,16842
XCE,float64,4604,21.46,16841
MK_CPY,str,2496,11.64,143
CTK_MGRNU,str,1265,5.90,2157
CAE_YMD,float64,890,4.15,2485


,MGRNU,A062_MGRNU,DRN_CDE,ESB_YMD,CAE_YMD,XCE,YCE,MK_CPY,SL_NUM,WORK_CDE,...,SIXID,STAT_CDE,CTK_MGRNU,SD_MGRNU,HISID,POS,FRM_CDE,MNG_AGEN,XGEO,geometry
0,24-0000011058,02-0000180110,0.0,20151026.0,20230102.0,187968.300000,549055.306244,대한신호,NaN,6,...,1264566.0,1.0,2023-0101-005,24-011058,15786,1.0,2.0,NaN,oracle.sql.STRUCT@15216338,POINT (187968.299999927 549055.306243806)
1,24-0000011081,02-0000180117,0.0,20140808.0,20210720.0,186876.700000,551491.393744,대한신호,NaN,6,...,1249343.0,1.0,2021-0101-035,24-011081,15809,1.0,2.0,NaN,oracle.sql.STRUCT@3aa2e0a8,POINT (186876.699999921 551491.393743825)
2,24-0000013895,02-0000178874,180.0,20250409.0,20250409.0,185617.877085,548627.649122,대한신호,NaN,1,...,12138710.0,1.0,2025-0101-051,24-013895,18623,1.0,2.0,NaN,oracle.sql.STRUCT@7fdf03d3,POINT (185617.877084836 548627.649122097)


## 4. 음향 신호기 코드 해석표

음향 신호기 데이터의 코드값을 지도 툴팁에 표시할 수 있는 의미 있는 문자열로 변환하기 위한 매핑표입니다.

In [4]:
# 사용자가 제공한 음향신호기 코드표
signal_kind_map = {1: '버튼식'}
facility_status_map = {1: '양호', 2: '파손', 3: '도색', 4: '노후'}
view_map = {1: '안보기', 2: '보기'}
construction_map = {
    1: '신설', 2: '교체', 3: '제거', 4: '도색', 5: '재도색',
    6: '이설', 7: '차광막교체', 8: '좌대', 9: '철거 후 재설치', 10: '신설(철거도색)'
}
work_map = {
    1: '자료입력', 2: '자료입력완료', 3: '승인대기중', 4: '부분완료', 5: '위치수정요망',
    6: '검증완료', 7: '승인', 8: '경찰청승인', 9: '시설물수정완료', 10: '내용수정요망',
    11: '수정요망', 12: '센터입력중', 13: '중간저장', 14: '반려'
}
direction_degree_map = {1: 0, 2: 45, 3: 90, 4: 135, 6: 180, 7: 225, 8: 270, 9: 315}

codebook_rows = []
for field, mapping in {
    'A073_KND_C': signal_kind_map, 'STAT_CDE': facility_status_map, 'VIEW_CDE': view_map,
    'FRM_CDE': construction_map, 'WORK_CDE': work_map, 'DRN_CDE': direction_degree_map,
}.items():
    codebook_rows.extend({'컬럼': field, '코드': f'{code:03d}', '의미': meaning} for code, meaning in mapping.items())
audible_codebook = pd.DataFrame(codebook_rows)
display(audible_codebook)

,컬럼,코드,의미
0,A073_KND_C,001,버튼식
1,STAT_CDE,001,양호
2,STAT_CDE,002,파손
3,STAT_CDE,003,도색
4,STAT_CDE,004,노후
5,VIEW_CDE,001,안보기
6,VIEW_CDE,002,보기
7,FRM_CDE,001,신설
8,FRM_CDE,002,교체
9,FRM_CDE,003,제거


## 5. 음향 신호기 표시용 컬럼 생성

시설 상태, 방향, 작업 단계 등 지도 표시와 툴팁에 필요한 파생 컬럼을 추가합니다.

In [5]:
def map_code(series, mapping):
    numeric = pd.to_numeric(series, errors='coerce').astype('Int64')
    return numeric.map(mapping).fillna('미상')

audible_enriched = audible.copy()
audible_enriched['음향신호기종류'] = map_code(audible_enriched['A073_KND_C'], signal_kind_map)
audible_enriched['시설물상태'] = map_code(audible_enriched['STAT_CDE'], facility_status_map)
audible_enriched['표출구분'] = map_code(audible_enriched['VIEW_CDE'], view_map)
audible_enriched['공사형태'] = map_code(audible_enriched['FRM_CDE'], construction_map)
audible_enriched['작업단계'] = map_code(audible_enriched['WORK_CDE'], work_map)
audible_enriched['방향도'] = map_code(audible_enriched['DRN_CDE'], direction_degree_map)

# 난이도 지수 설계를 위한 상태별 '지원 효과'와 '유지보수 경고' 초안
support_score_map = {'양호': 1.0, '파손': 0.0, '도색': 0.5, '노후': 0.5, '미상': pd.NA}
warning_map = {
    '양호': '경고 없음', '파손': '지원 효과 제외 + 유지보수 위험',
    '도색': '작동 여부 확인 필요', '노후': '작동·유지관리 상태 확인 필요', '미상': '상태 미확인'
}
audible_enriched['상태_지원점수_제안'] = audible_enriched['시설물상태'].map(support_score_map)
audible_enriched['상태_경고'] = audible_enriched['시설물상태'].map(warning_map)

status_summary = (audible_enriched.groupby(['시설물상태', '표출구분', '작업단계'], dropna=False)
                  .size().rename('장치수').reset_index().sort_values('장치수', ascending=False))
display(status_summary)

,시설물상태,표출구분,작업단계,장치수
3,양호,보기,자료입력,18933
1,양호,보기,검증완료,2322
2,양호,보기,승인대기중,171
0,미상,보기,자료입력,25


## 6. 장애인 편의시설 데이터 준비

지도에 표시할 장애인 편의시설 데이터를 불러옵니다.

In [6]:
# 2) 장애인편의시설 목록
facility = read_csv_flexible(DATASETS['facility'])
facility_summary = profile(facility, '장애인편의시설')

서울시_장애인편의시설_목록정보(한국사회보장정보원)_.csv: encoding=cp949, rows=29,153, cols=12

[장애인편의시설] shape=(29153, 12), duplicate rows=0


,dtype,missing_n,missing_pct,nunique
시설대표자성명,float64,29153,100.0,0
순번,int64,0,0.0,29153
시설기본주소,str,0,0.0,28751
시설ID,str,0,0.0,28684
시설위도,float64,0,0.0,23589
시설경도,float64,0,0.0,23492
시설명,str,0,0.0,19208
설립일자,int64,0,0.0,6116
시설유형,str,0,0.0,67
영업상태구분코드,str,0,0.0,1


,설립일자,순번,시설위도,시설경도,시설명,시설대표자성명,시설유형,시설기본주소,영업상태구분코드,영업상태구분명,복지로관리시설구분코드,시설ID
0,20120130,20155836847,0.000000,0.000000,서울강동초등학교 다목적강당,NaN,초등학교,서울 강동구 상암로14길 77,Y,영업,17,1174011000-1-03060000
1,20010910,20155836848,37.547399,127.146421,명성교회 월드글로리아센터,NaN,대피소,서울 강동구 양재대로134길 52(명일동),Y,영업,17,1174010100-1-03400001
2,20170425,20155836852,0.000000,0.000000,수유역두산위브아파트1-번1동,NaN,대피소,서울 강북구 덕릉로41길 12,Y,영업,17,KODIX-SL-0000001


## 7. 사고 데이터 준비

지도에 표시할 보행자 사고 다발지역 데이터를 불러옵니다.

In [7]:
# 4) 보행자 사고 다발지역
accident = read_csv_flexible(DATASETS['accident'])
accident_summary = profile(accident, '보행자 사고 다발지역')

보행자_사고_다발지역.csv: encoding=cp949, rows=2,402, cols=15

[보행자 사고 다발지역] shape=(2402, 15), duplicate rows=0


,dtype,missing_n,missing_pct,nunique
사고다발지fid,int64,0,0.0,2402
다발지역폴리곤,str,0,0.0,2328
경도,float64,0,0.0,2297
위도,float64,0,0.0,2297
지점명,str,0,0.0,1900
시도시군구명,str,0,0.0,1270
지점코드,int64,0,0.0,788
법정동코드,int64,0,0.0,713
사상자수,int64,0,0.0,21
중상자수,int64,0,0.0,19


,사고다발지fid,사고다발지id,법정동코드,지점코드,시도시군구명,지점명,사고건수,사상자수,사망자수,중상자수,경상자수,부상신고자수,경도,위도,다발지역폴리곤
0,6778887,2022040,1111017100,11110001,서울특별시 종로구1,서울특별시 종로구 명륜2가(성균관대입구교차로 부근),9,10,1,9,0,0,126.998780,37.582940,"{""type"":""Polygon"",""coordinates"":[[[126.9996779..."
1,6779548,2022040,1111013700,11110002,서울특별시 종로구2,서울특별시 종로구 낙원동(낙원상가 부근),8,8,0,8,0,0,126.988435,37.571899,"{""type"":""Polygon"",""coordinates"":[[[126.9893336..."
2,6779517,2022040,1111011900,11110003,서울특별시 종로구3,서울특별시 종로구 세종로(세종로180 부근),7,8,0,8,0,0,126.976899,37.569865,"{""type"":""Polygon"",""coordinates"":[[[126.9777971..."


## 8. 공간 데이터 변환과 반경 필터링

각 원천 데이터의 좌표를 지도용 좌표계로 정리하고, 설정된 기준점과의 거리를 계산합니다. `RADIUS_M` 이내의 데이터만 지도 레이어로 사용합니다.

In [8]:
# EPSG:5186 = KGD2002 / Central Belt 2010 (음향신호기 shapefile .prj와 일치)
audible_enriched['XCE'] = pd.to_numeric(audible_enriched['XCE'], errors='coerce')
audible_enriched['YCE'] = pd.to_numeric(audible_enriched['YCE'], errors='coerce')
audible_geo = gpd.GeoDataFrame(
    audible_enriched.dropna(subset=['XCE', 'YCE']).copy(),
    geometry=gpd.points_from_xy(audible_enriched.dropna(subset=['XCE', 'YCE'])['XCE'], audible_enriched.dropna(subset=['XCE', 'YCE'])['YCE']),
    crs='EPSG:5186',
).to_crs('EPSG:4326')

facility['시설위도'] = pd.to_numeric(facility['시설위도'], errors='coerce')
facility['시설경도'] = pd.to_numeric(facility['시설경도'], errors='coerce')
facility_geo = gpd.GeoDataFrame(
    facility.loc[facility['시설위도'].between(33, 39) & facility['시설경도'].between(124, 132)].copy(),
    geometry=gpd.points_from_xy(
        facility.loc[facility['시설위도'].between(33, 39) & facility['시설경도'].between(124, 132), '시설경도'],
        facility.loc[facility['시설위도'].between(33, 39) & facility['시설경도'].between(124, 132), '시설위도'],
    ),
    crs='EPSG:4326',
)

accident['위도'] = pd.to_numeric(accident['위도'], errors='coerce')
accident['경도'] = pd.to_numeric(accident['경도'], errors='coerce')
accident_geo = gpd.GeoDataFrame(
    accident.dropna(subset=['위도', '경도']).copy(),
    geometry=gpd.points_from_xy(accident.dropna(subset=['위도', '경도'])['경도'], accident.dropna(subset=['위도', '경도'])['위도']),
    crs='EPSG:4326',
)

center = gpd.GeoDataFrame(geometry=[Point(CENTER_LON, CENTER_LAT)], crs='EPSG:4326').to_crs('EPSG:5186')
def add_center_distance(gdf):
    projected = gdf.to_crs('EPSG:5186').copy()
    projected['center_distance_m'] = projected.geometry.distance(center.geometry.iloc[0])
    return projected.to_crs('EPSG:4326')

audible_geo = add_center_distance(audible_geo)
facility_geo = add_center_distance(facility_geo)
accident_geo = add_center_distance(accident_geo)

nearby_audible = audible_geo.query('center_distance_m <= @RADIUS_M').copy()
nearby_facility = facility_geo.query('center_distance_m <= @RADIUS_M').copy()
nearby_accident = accident_geo.query('center_distance_m <= @RADIUS_M').copy()

pd.DataFrame({
    'dataset': ['음향신호기', '장애인편의시설', '보행자 사고 다발지역'],
    'valid_coordinate_rows': [len(audible_geo), len(facility_geo), len(accident_geo)],
    f'within_{RADIUS_M}m': [len(nearby_audible), len(nearby_facility), len(nearby_accident)],
})

,dataset,valid_coordinate_rows,within_800m
0,음향신호기,16847,69
1,장애인편의시설,24075,216
2,보행자 사고 다발지역,2402,5


## 9. 보조 데이터 지도 시각화

설정된 기준점을 중심으로 반경 원을 그리고, 음향 신호기·장애인 편의시설·사고 지점·사고 다발지역을 지도에 표시합니다.

In [9]:
m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15, tiles='OpenStreetMap')
folium.Marker([CENTER_LAT, CENTER_LON], tooltip='분석 기준점', icon=folium.Icon(color='red')).add_to(m)
folium.Circle([CENTER_LAT, CENTER_LON], radius=RADIUS_M, color='#2563eb', fill=False, tooltip='800m 참고 범위').add_to(m)

signal_color = {'양호': '#16a34a', '파손': '#dc2626', '도색': '#f59e0b', '노후': '#7c3aed', '미상': '#6b7280'}
for _, row in nearby_audible.iterrows():
    status = row['시설물상태']
    tooltip = f"음향신호기 | {status} | 방향 {row['방향도']}도 | {row['상태_경고']}"
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x], radius=4, color=signal_color.get(status, '#6b7280'),
        fill=True, fill_opacity=0.9, tooltip=tooltip
    ).add_to(m)

for _, row in nearby_facility.iterrows():
    label = f"{row['시설명']} ({row['시설유형']})"
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=3, color='#7c3aed', fill=True, tooltip=label).add_to(m)

for _, row in nearby_accident.iterrows():
    tooltip = f"{row['지점명']} | 사고 {row['사고건수']}건"
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=5, color='#dc2626', fill=True, fill_opacity=0.8, tooltip=tooltip).add_to(m)
    try:
        polygon = shape(json.loads(row['다발지역폴리곤']))
        folium.GeoJson(polygon.__geo_interface__, style_function=lambda _: {'color': '#dc2626', 'weight': 1, 'fillOpacity': 0.12}).add_to(m)
    except (TypeError, ValueError, KeyError):
        pass

from IPython.display import HTML, IFrame
HTML(m.get_root().render())
IFrame(src='http://localhost:8000/', width='100%', height=700)

## 10. 보행 네트워크 우선순위 지도

보행 네트워크와 반경 내 보조 데이터를 결합해 edge별 우선순위를 계산하고, 결과를 색상으로 시각화합니다.

In [10]:
import networkx as nx
import numpy as np
from shapely.geometry import LineString

# 경사도가 포함된 보행 네트워크를 불러와 edge GeoDataFrame으로 변환
graph_path = PROJECT_DIR / 'data' / 'pedestrian_graph_with_slope.graphml'
G = nx.read_graphml(graph_path)
edge_rows = []
for u, v, key, data in G.edges(keys=True, data=True):
    u_node, v_node = G.nodes[u], G.nodes[v]
    if not all(k in u_node and k in v_node for k in ('x', 'y')):
        continue
    record = dict(data)
    record.update({
        'edge_id': f'{u}|{v}|{key}',
        'u': u, 'v': v, 'key': key,
        'geometry': LineString([(float(u_node['x']), float(u_node['y'])),
                                 (float(v_node['x']), float(v_node['y']))]),
    })
    edge_rows.append(record)

edges = gpd.GeoDataFrame(edge_rows, geometry='geometry', crs='EPSG:4326')
edges['length_m'] = pd.to_numeric(edges.get('length'), errors='coerce')
edges['slope_pct'] = pd.to_numeric(edges.get('slope_pct'), errors='coerce').fillna(0)
edges_proj = edges.to_crs('EPSG:5186')

def attach_nearest_count(edge_frame, points, prefix, max_distance_m=30):
    """가까운 점 데이터의 개수와 최소 거리를 edge에 결합한다."""
    points_proj = points[['geometry']].dropna().to_crs('EPSG:5186')
    joined = gpd.sjoin_nearest(
        edge_frame[['edge_id', 'geometry']], points_proj,
        how='left', max_distance=max_distance_m,
        distance_col=f'{prefix}_distance_m',
    )
    matched = joined.dropna(subset=['index_right'])
    if matched.empty:
        summary = pd.DataFrame(index=edge_frame['edge_id'])
    else:
        summary = matched.groupby('edge_id').agg(
            **{
                f'{prefix}_count': ('index_right', 'size'),
                f'{prefix}_distance_m': (f'{prefix}_distance_m', 'min'),
            }
        )
    result = edge_frame.set_index('edge_id').join(summary).reset_index()
    result[f'{prefix}_count'] = result[f'{prefix}_count'].fillna(0).astype(int)
    result[f'{prefix}_distance_m'] = result[f'{prefix}_distance_m'].fillna(np.inf)
    return result

# 앞 단계 셀을 건너뛰고 이 셀만 실행해도 분석 대상 범위를 준비한다.
if 'nearby_audible' not in globals():
    nearby_audible = audible_geo.query('center_distance_m <= @RADIUS_M').copy()
if 'nearby_facility' not in globals():
    nearby_facility = facility_geo.query('center_distance_m <= @RADIUS_M').copy()
if 'nearby_accident' not in globals():
    nearby_accident = accident_geo.query('center_distance_m <= @RADIUS_M').copy()

edge_context = edges_proj
edge_context = attach_nearest_count(edge_context, nearby_audible, 'audible')
edge_context = attach_nearest_count(edge_context, nearby_facility, 'facility')
edge_context = attach_nearest_count(edge_context, nearby_accident, 'accident')

# 설명용 MVP 점수: 숫자가 높을수록 개선 우선순위가 높은 구간
edge_context['slope_component'] = np.clip(edge_context['slope_pct'].abs() / 8, 0, 1) * 4
edge_context['accident_risk_component'] = np.clip(edge_context['accident_count'] / 3, 0, 3)
edge_context['audible_support_component'] = np.where(edge_context['audible_count'] > 0, 1.5, 0)
edge_context['facility_support_component'] = np.clip(edge_context['facility_count'] / 3, 0, 1)
edge_context['priority_score'] = (
    edge_context['slope_component']
    + edge_context['accident_risk_component']
    - edge_context['audible_support_component']
    - edge_context['facility_support_component']
)

display(edge_context[['edge_id', 'length_m', 'slope_pct', 'audible_count',
                      'facility_count', 'accident_count', 'priority_score']]
        .sort_values('priority_score', ascending=False).head(20))

# 최종 결과: 우선순위가 높은 보행 edge를 붉게 표시하는 지도
from branca.colormap import linear

edge_map = edge_context[['edge_id', 'length_m', 'slope_pct', 'audible_count',
                        'facility_count', 'accident_count', 'priority_score',
                        'geometry']].to_crs('EPSG:4326')
score_min = float(edge_map['priority_score'].min())
score_max = float(edge_map['priority_score'].max())
if score_min == score_max:
    score_max = score_min + 1
score_color = linear.YlOrRd_09.scale(score_min, score_max)

priority_map = folium.Map(
    location=[CENTER_LAT, CENTER_LON], zoom_start=15, tiles='OpenStreetMap'
)
folium.Marker([CENTER_LAT, CENTER_LON], tooltip='분석 기준점',
              icon=folium.Icon(color='blue')).add_to(priority_map)
folium.Circle([CENTER_LAT, CENTER_LON], radius=RADIUS_M, color='#2563eb',
              fill=False, tooltip='분석 범위 800m').add_to(priority_map)

folium.GeoJson(
    edge_map.to_json(),
    name='보행 edge 우선순위',
    style_function=lambda feature: {
        'color': score_color(feature['properties']['priority_score']),
        'weight': 5, 'opacity': 0.85,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['edge_id', 'slope_pct', 'audible_count', 'facility_count',
                'accident_count', 'priority_score'],
        aliases=['edge', '경사도(%)', '음향 신호기 수', '편의시설 수',
                 '사고 지점 수', '개선 우선순위'],
        localize=True, sticky=False, labels=True,
    ),
).add_to(priority_map)

for _, row in nearby_audible.to_crs('EPSG:4326').iterrows():
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=4,
                        color='#16a34a', fill=True, tooltip='음향 신호기').add_to(priority_map)
for _, row in nearby_facility.to_crs('EPSG:4326').iterrows():
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=3,
                        color='#7c3aed', fill=True, tooltip='장애인 편의시설').add_to(priority_map)
for _, row in nearby_accident.to_crs('EPSG:4326').iterrows():
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=5,
                        color='#dc2626', fill=True, tooltip='사고 지점').add_to(priority_map)

score_color.caption = '보행 edge 개선 우선순위(높을수록 우선)'
score_color.add_to(priority_map)
from IPython.display import HTML, IFrame
HTML(priority_map.get_root().render())
IFrame(src='http://localhost:8000/', width='100%', height=700)

,edge_id,length_m,slope_pct,audible_count,facility_count,accident_count,priority_score
1300,8757685953|436831584|0,13.266215,11.42,0,0,0,4.0
13,436831584|570912259|0,21.752041,8.02,0,0,0,4.0
1289,8757685950|5826295509|0,11.743413,8.73,0,0,0,4.0
456,3337436851|3337436844|0,187.425927,12.19,0,0,0,4.0
908,4621488498|4621488496|0,26.017764,8.72,0,0,0,4.0
906,4621488496|4621488498|0,26.017764,8.72,0,0,0,4.0
944,4621496734|10265961199|0,25.449194,15.41,0,0,0,4.0
14,436839948|436839950|0,37.580110,8.60,0,0,0,4.0
449,3337436844|13281452472|0,10.892126,8.72,0,0,0,4.0
1191,6964092882|6964092879|1,91.433835,17.91,0,0,0,4.0
